# High-rise 03 - Per-floor force / moment coefficients (Cf, Cm)

Read the Cp time series from notebook 02, slice the body by floor z-edges, and sum
the per-triangle force / moment contributions per floor (explicit reference-area
normalisation). Writes per-floor Cf/Cm profiles to `debug/` and a load table to
`deliverables/`.

Uses the same case + mesh as notebook 02. For the fixture the floor edges come from
the mesh z-range; for a real case they come from the case_data `HEIGHTS`.

In [ ]:
import os
import pathlib

import numpy as np  # noqa: F401  (used across later cells)

import matplotlib

matplotlib.use("Agg")  # headless: notebooks write files, they do not display

from cfdmod import inflow_report, mesh_field, plot_config  # noqa: E402
from cfdmod.building import (  # noqa: E402
    BuildingCase,
    cf_per_floor,
    cm_per_floor,
    cp_from_pressure,
    example_building_case,
    example_building_structure,
    floor_accelerations,
    floor_load_source,
    peak_response_table,
    peak_value,
    plot_floor_mass,
    plot_mode_shape,
    plot_natural_frequencies,
    solve_building_response,
    structure_from_csvs,
)
from cfdmod.report import DebugWriter  # noqa: E402


def _find_repo(start: pathlib.Path) -> pathlib.Path:
    p = start.resolve()
    while p != p.parent:
        if (p / "pyproject.toml").exists():
            return p
        p = p.parent
    return start.resolve()


REPO = _find_repo(pathlib.Path.cwd())

plot_config.apply_style()

FIX = REPO / "fixtures" / "tests"
OUTPUT_BASE = pathlib.Path(
    os.environ.get("CFDMOD_HR_OUTPUT_BASE", REPO / "examples" / "high_rise" / "_run")
)
VERSION = os.environ.get("CFDMOD_HR_VERSION", "example")
print("REPO:", REPO)
print("OUTPUT_BASE:", OUTPUT_BASE, "| VERSION:", VERSION)

In [ ]:
from cfdmod.adapters.xdmf_h5 import XdmfH5Storage

# --- config -------------------------------------------------------------
MESH = pathlib.Path(
    os.environ.get("CFDMOD_HR_MESH", FIX / "pressure" / "galpao" / "galpao.normalized.lnas")
)
CASE_DATA = os.environ.get("CFDMOD_HR_CASE_DATA")
PARAMS = os.environ.get("CFDMOD_HR_PARAMS", "params_cat3.yaml")
if CASE_DATA:
    case = BuildingCase.from_case_data(pathlib.Path(CASE_DATA), PARAMS)
else:
    case = example_building_case(MESH)

ARTIFACTS = OUTPUT_BASE / "artifacts" / VERSION
cp = XdmfH5Storage(ARTIFACTS).read_data_source(pathlib.Path("cp.time_series"))
print(f"cp: {cp.n_elements} elements, {cp.time.n_timesteps} steps | {case.n_floors} floors")

In [ ]:
# --- per-floor Cf / Cm --------------------------------------------------
cf = cf_per_floor(cp, str(MESH), case, directions=("x", "y"))
cm = cm_per_floor(cp, str(MESH), case, directions=("z",))

# Floor mid-heights for plotting.
edges = np.asarray(case.floor_heights)
z_mid = 0.5 * (edges[:-1] + edges[1:])
cfx_mean = np.nanmean(cf.fields.read("cf_x"), axis=1)
cfy_mean = np.nanmean(cf.fields.read("cf_y"), axis=1)
cmz_mean = np.nanmean(cm.fields.read("cm_z"), axis=1)
print("per-floor mean Cf_x:", np.round(cfx_mean, 4))

In [ ]:
import pandas as pd

# --- figures + load table ----------------------------------------------
dbg = DebugWriter(OUTPUT_BASE, stage="cf", version=VERSION)
fig, ax = plot_config.new_axes(xlabel="mean coefficient [-]", ylabel="z [m]", title="Per-floor Cf / Cm")
ax.plot(cfx_mean, z_mid, "-o", ms=3, label="Cf_x")
ax.plot(cfy_mean, z_mid, "-o", ms=3, label="Cf_y")
ax.plot(cmz_mean, z_mid, "-s", ms=3, label="Cm_z")
ax.legend()
dbg.savefig(fig, "per_floor_coefficients.png")
plot_config.close(fig)

table = pd.DataFrame(
    {
        "floor": np.arange(len(z_mid)),
        "z_mid": z_mid,
        "cf_x_mean": cfx_mean,
        "cf_y_mean": cfy_mean,
        "cm_z_mean": cmz_mean,
    }
)
table.to_csv(dbg.deliverable_path("per_floor_loads.csv"), index=False)
table